# Explore Wikispeedia Paths and Graph Data

In [1]:
import numpy as np
import pandas as pd
import os
from urllib.parse import unquote
import random
from time import sleep
import pickle
import json
import copy
import re
import regex

%load_ext autoreload
%autoreload 2

random.seed(42)

PATH_IN = '../data/wikispeedia/wikispeedia_paths-and-graph/'

## Create a DF with the links and paths

In [2]:
# Wikipedia articles as nodes in a graph
# ex: 'Aristotle' -> True if it is an article
fnodes = open(os.path.join(PATH_IN, 'articles.tsv'))
nodes = {}
for i, line in enumerate(fnodes):
    line = line.strip()
    if '#' in line or line == '':
        continue
    node = unquote(line)
    nodes[node]=True


In [3]:
len(nodes)

4604

In [4]:
# Links between Wikipedia articles
# ex: 'Aristotle' -> ['Philosophy', 'Metaphysics', ...]
flinks = open(os.path.join(PATH_IN, 'links.tsv'))
links = {}
for i, line in enumerate(flinks):
    line = line.strip()
    if '#' in line or line == '':
        continue
    line = line.split('\t')
    src = unquote(line[0]); tgt = unquote(line[1])
    if src not in links:
        links[src] = [tgt]
    else:
        links[src].append(tgt)

In [5]:
links['Aristotle'][0:10]

['Aesthetics',
 'Alexander_the_Great',
 'Anatomy',
 'Ancient_Greece',
 'Astronomy',
 'Athens',
 'Bertrand_Russell',
 'Biology',
 'Christianity',
 'Economics']

In [6]:
# create a dictionary with article and links
article_links = {}
for article in links:
    article_links[article] = ";".join(links[article])

# create a dataframe with article and links
df_article_links = pd.DataFrame(list(article_links.items()), columns=['article', 'links'])

df_article_links.to_csv(os.path.join(PATH_IN, 'article_links.tsv'), sep='\t', index=False)


In [7]:
pairs_finished = {}; pairs_unfinished = {}; paths_real = []

In [8]:
# Paths taken by users between Wikipedia articles
# fields: source, target, path, length, is_finished
fpaths_finished = open(os.path.join(PATH_IN, 'paths_finished.tsv'))
for i, line in enumerate(fpaths_finished):
    line = line.strip()
    if '#' in line or line == '':
        continue
    line = line.split('\t')
    path = line[3].split(';')
    src = unquote(path[0]); tgt = unquote(path[-1])
    path_unquote = []
    for article in path:
        path_unquote.append(unquote(article))
    if (src,tgt) not in pairs_finished:
        pairs_finished[(src,tgt)] = True
    paths_real.append([src,tgt,";".join(path_unquote),len(path_unquote)-1,1])

fpaths_unfinished = open(os.path.join(PATH_IN, 'paths_unfinished.tsv'))
for i, line in enumerate(fpaths_unfinished):
    line = line.strip()
    if '#' in line or line == '':
        continue
    line = line.split('\t')
    path = line[3].split(';')
    src = unquote(path[0]); tgt = unquote(line[4])
    path_unquote = []
    for article in path:
        path_unquote.append(unquote(article))
    if (src,tgt) not in pairs_unfinished and (src,tgt) not in pairs_finished:
        pairs_unfinished[(src,tgt)] = True
    paths_real.append([src,tgt,";".join(path_unquote),len(path_unquote)-1,0])

In [9]:
paths_real_df = pd.DataFrame(paths_real, columns=['src', 'tgt', 'path', 'path_len', 'is_path_finished'])
# save to file
paths_real_df.to_csv(os.path.join(PATH_IN, 'paths_real.tsv'), index=False, sep='\t')
paths_real_df.head(5)

,src,tgt,path,path_len,is_path_finished
0,14th_century,African_slave_trade,14th_century;15th_century;16th_century;Pacific...,8,1
1,14th_century,African_slave_trade,14th_century;Europe;Africa;Atlantic_slave_trad...,4,1
2,14th_century,African_slave_trade,14th_century;Niger;Nigeria;British_Empire;Slav...,7,1
3,14th_century,Greece,14th_century;Renaissance;Ancient_Greece;Greece,3,1
4,14th_century,John_F._Kennedy,14th_century;Italy;Roman_Catholic_Church;HIV;R...,6,1


In [10]:
# see some samples
# we will use paths_real_finished_sample to prompt the LLM
paths_real_finished_sample = paths_real_df.query('is_path_finished == 1 and path_len <= 10').sample(n=100, random_state=42)
paths_real_unfinished_sample = paths_real_df.query('is_path_finished == 0').sample(n=10, random_state=42)

In [11]:
paths_real_finished_sample.head(5)

,src,tgt,path,path_len,is_path_finished
5715,Jurassic,Cologne,Jurassic;Germany;Cologne,2,1
14457,Brain,Telephone,Brain;Computer_science;Society;<;Internet;Info...,7,1
29624,Honey,"September_11,_2001_attacks","Honey;United_States;September_11,_2001_attacks",2,1
39628,Great_Yarmouth,Salsa_music,Great_Yarmouth;World_War_II;United_States;Mexi...,5,1
32444,Anna_Karenina,Age_of_Enlightenment,Anna_Karenina;Saint_Petersburg;Jew;Age_of_Enli...,3,1


In [12]:
# Create a DF with path demonstrations, to use for few-shot learning
# fields: src, tgt, output
path_demonstrations = []
paths_real_sample = paths_real_df.query('is_path_finished == 1').sample(n=20, random_state=101)[['src', 'tgt', 'path']].values.tolist()
for path in paths_real_sample:
    path_obj = {}
    path_obj['src'] = path[0]; path_obj['tgt'] = path[1]
    path_obj['output'] = path[2]
    path_demonstrations.append(path_obj)
path_demonstrations

[{'src': 'Ottoman_Empire',
  'tgt': 'Rosetta_Stone',
  'output': 'Ottoman_Empire;Egypt;Ancient_Egypt;Rosetta_Stone'},
 {'src': 'Thomas_Cranmer',
  'tgt': 'Computer_and_video_games',
  'output': 'Thomas_Cranmer;William_Shakespeare;England;Time_zone;Telecommunication;Internet;Microsoft;Television;Computer_and_video_games'},
 {'src': 'American_Revolutionary_War',
  'tgt': 'Bison',
  'output': 'American_Revolutionary_War;United_States;Yellowstone_National_Park;Cattle;Bison'},
 {'src': 'Omega-3_fatty_acid',
  'tgt': 'Chew_Valley',
  'output': 'Omega-3_fatty_acid;Schizophrenia;United_Kingdom;England;Bristol;<;History_of_Anglo-Saxon_England;<;<;England;River_Severn;<;<;England;John_Locke;Chew_Magna;Chew_Valley'},
 {'src': 'The_Supremes',
  'tgt': 'Iron_Age',
  'output': 'The_Supremes;Chicago;Industry;Industrial_Revolution;<;Steel;Iron;<;Bronze_Age;Iron_Age'},
 {'src': 'Owl', 'tgt': 'Penguin', 'output': 'Owl;Antarctica;Penguin'},
 {'src': 'Weed',
  'tgt': 'Clifton_Suspension_Bridge',
  'output

In [13]:
# Path demonstrations with clicks in candidate links, for few-shot learning
# fields: src, tgt, cands, output
click_demonstrations = []
paths_real_sample = paths_real_df.query('is_path_finished == 1').sample(n=20, random_state=101)[['src', 'tgt', 'path']].values.tolist()
for path in paths_real_sample:
    path_obj = {}
    path_obj['src'] = path[0]; path_obj['tgt'] = path[1]
    path_obj['cands'] = ";".join(links[path_obj['src']])
    path_obj['output'] = path[2].split(";")[1]
    click_demonstrations.append(path_obj)
click_demonstrations

[{'src': 'Ottoman_Empire',
  'tgt': 'Rosetta_Stone',
  'cands': "15th_century;16th_century;17th_century;20th_century;Abbasid;Albania;Algeria;Arabic_language;Armenia;Asia;Atlantic_Ocean;Austria;Azerbaijan;Baghdad;Baroque;Beirut;Black_Sea;Bulgaria;Byzantine_Empire;Calligraphy;Caspian_Sea;Christopher_Columbus;Constitutional_monarchy;Crimean_War;Currency;Cyprus;Egypt;England;Europe;Folk_music;France;Greece;Greek_War_of_Independence;Guild;Habsburg_Spain;Holy_Roman_Empire;Indian_Ocean;Islam;Istanbul;Jerusalem;Johannes_Gutenberg;Libya;List_of_countries_by_system_of_government;Marco_Polo;Mediterranean_Sea;Mesopotamia;Middle_Ages;Middle_East;Moldova;Monarchy;Mongol_Empire;Montenegro;Muhammad;Musical_instrument;Nationalism;North_Africa;Persian_Gulf;Piano;Poland;Portugal;Qur'an;Red_Sea;Rococo;Roman_Empire;Romania;Russia;Russian_Revolution_of_1917;Sarajevo;Serbia;Siege;Slavery;Slovakia;Spanish_Inquisition;Sudan;Suleiman_the_Magnificent;Sultan;The_Holocaust;Tunis;Turkey;Ukraine;University;Vienna;Wo

In [14]:
# Path demonstrations with clicks in candidate links and memory of current path
# , for few-shot learning
# fields: src, tgt, cands, path_so_far, output
click_demonstrations_with_memory = []
paths_real_sample = paths_real_df.query('is_path_finished == 1').sample(n=20, random_state=101)[['src', 'tgt', 'path']].values.tolist()
random.seed(42)
for i, path in enumerate(paths_real_sample):
    path_obj = {}
    path_obj['src'] = path[0]; path_obj['tgt'] = path[1]
    path_list = path[2].split(";")
    cur_idx = random.randint(0,len(path_list)-2)
    if i==19:
        cur_idx=0
    # print(cur_idx, len(path_list))
    path_obj['cur'] = path_list[cur_idx]
    if path_obj['cur'] == "<":
        # print("Cur=<", path)
        path_obj['cur'] = 'United_Kingdom'
        path_obj['cands'] = ";".join(links['United_Kingdom'])
        path_obj['path_so_far'] = ";".join(['Omega-3_fatty_acid', 'Schizophrenia', 'United_Kingdom', 'England', 'Bristol', 'England', 'History_of_Anglo-Saxon_England', 'England', 'United_Kingdom'])
    else:
        path_obj['cands'] = ";".join(links[path_obj['cur']])
        if i==17:
            path_obj['path_so_far'] = ";".join(['Samuel_Beckett', 'Miami,_Florida', 'Hurricane_Andrew', 'Tropical_cyclone', 'Hurricane_Andrew', 'Miami,_Florida', 'Hurricane_Andrew', 'Miami,_Florida', 'Hurricane_Andrew'])    
        else:
            path_obj['path_so_far'] = ";".join(path_list[:cur_idx+1])
    if path_list[cur_idx+1] == "<":
        # print("Cur+1=<", path)
        if path_obj['cur']=='Industrial_Revolution':
            path_obj['output'] = "Iron"
        elif path_obj['cur']=='Jew':
            path_obj['output'] = "Judaism"
    else:
        path_obj['output'] = path_list[cur_idx+1]
    click_demonstrations_with_memory.append(path_obj)
click_demonstrations_with_memory

[{'src': 'Ottoman_Empire',
  'tgt': 'Rosetta_Stone',
  'cur': 'Ancient_Egypt',
  'cands': '15th_century;4th_century;Africa;Agriculture;Alexander_the_Great;Ancient_Greece;Ancient_Rome;Anemia;Anno_Domini;Archaeology;Beekeeping;Beer;Berber_languages;Board_game;Boat;Carbon;Cereal;Chess;Christianity;Civilization;Copper;Culture;Currency;East_Africa;Egypt;Egyptian_pyramids;Electricity;Empiricism;Engineering;Fishing;Geology;Geometry;Glass;Granite;Great_Pyramid_of_Giza;Herodotus;Homer;Horn_of_Africa;Iron;Iron_Age;Irrigation;Islam;Literature;Mathematics;Mediterranean_Sea;Middle_Ages;Mining;Mummy;Nile;North_Africa;Odysseus;Property;Pyramid;Red_Sea;Religion;Roman_Empire;Rosetta_Stone;Sahara;Society;Steel;Sudan;Suez_Canal;Turquoise;Virgil;Vowel;Wine;Writing',
  'path_so_far': 'Ottoman_Empire;Egypt;Ancient_Egypt',
  'output': 'Rosetta_Stone'},
 {'src': 'Thomas_Cranmer',
  'tgt': 'Computer_and_video_games',
  'cur': 'William_Shakespeare',
  'cands': 'Actor;Ben_Jonson;British_Empire;Employment;England

In [15]:
# Save the demonstrations to csv
df_path_demonstrations = pd.DataFrame(path_demonstrations)
df_path_demonstrations.to_csv(os.path.join(PATH_IN, 'path_demonstrations.tsv'), index=False, sep='\t')
df_click_demonstrations = pd.DataFrame(click_demonstrations)
df_click_demonstrations.to_csv(os.path.join(PATH_IN, 'click_demonstrations.tsv'), index=False, sep='\t')
df_click_demonstrations_with_memory = pd.DataFrame(click_demonstrations_with_memory)
df_click_demonstrations_with_memory.to_csv(os.path.join(PATH_IN, 'click_demonstrations_with_memory.tsv'), index=False, sep='\t')

# Testing Prompt Functions

In [82]:
from prompt_utils import *
from navigate_links_only import *
import os
import yaml
import sys
import pandas as pd
sys.path.append('src/')
# auto-reload the module
%load_ext autoreload
%autoreload 2
# load .yaml with prompts in configs
with open('configs/experiments/navigate_link_only_entire_path.yaml', 'r') as stream:
    prompts = yaml.safe_load(stream)['prompts']


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [92]:
path_demonstrations = get_path_demonstrations('../data/wikispeedia/wikispeedia_paths-and-graph/path_demonstrations.tsv')
messages = get_navigate_entire_path_prompt(**prompts,src="Thomas_Cranmer", tgt="Video_Games", path_demonstrations=path_demonstrations)
for m in messages:
    print(m)

Assistant is a expert JSON builder designed to assist the user in playing play the game of Wikispeedia. Assistant is familiar with the Wikipedia link structure and can help the user find the shortest path between two Wikipedia articles.
In Wikispeedia, given two Wikipedia articles, a source and a target, your goal is to reach the target article starting from the source article in as few clicks as possible. For the articles given this is always possible.

All of Assistant's communication is performed using this JSON format.

Here are some previous conversations between the Assistant and User:
User: Find the path between "Ottoman_Empire" and "Rosetta_Stone".
Assistant: 
{
"Path": "Ottoman_Empire;Egypt;Ancient_Egypt;Rosetta_Stone"
}
User: Find the path between "Thomas_Cranmer" and "Computer_and_video_games".
Assistant: 
{
"Path": "Thomas_Cranmer;William_Shakespeare;England;Time_zone;Telecommunication;Internet;Microsoft;Television;Computer_and_video_games"
}
User: Find the path between "Am

In [54]:
links = load_links(path = '../data/wikispeedia/wikispeedia_paths-and-graph/links.tsv')
links

TypeError: load_links() got an unexpected keyword argument 'path'

In [57]:
src = "Citizenship"; tgt = "Hebrew_calendar"
path = 'Citizenship;Nationalism;Hebrew_language;Jew;<;Jerusalem;<;Israel;<;Jew;<;Judaism;Hebrew_calendar'
correct_path = "The_Supremes;Chicago;Industry;Steel;Bronze_Age;Iron_Age"
check_path(src,tgt,path, links_file='../data/wikispeedia/wikispeedia_paths-and-graph/links.tsv')

['Citizenship', 'Nationalism', 'Hebrew_language', 'Jew', '<', 'Jerusalem', '<', 'Israel', '<', 'Jew', '<', 'Judaism', 'Hebrew_calendar']
['Citizenship', 'Nationalism', 'Hebrew_language', 'Jew', '<', 'Jerusalem', '<', 'Israel', '<', 'Jew', '<', 'Judaism', 'Hebrew_calendar']
['Citizenship', 'Nationalism', 'Hebrew_language', 'Jew', '<', 'Jerusalem', '<', 'Israel', '<', 'Jew', '<', 'Judaism', 'Hebrew_calendar']
['Citizenship', 'Nationalism', 'Hebrew_language', 'Jew', '<', 'Jerusalem', '<', 'Israel', '<', 'Jew', '<', 'Judaism', 'Hebrew_calendar']
['Citizenship', 'Nationalism', 'Hebrew_language', 'Jew', '<', 'Jerusalem', '<', 'Israel', '<', 'Jew', '<', 'Judaism', 'Hebrew_calendar']
['Citizenship', 'Nationalism', 'Hebrew_language', 'Jerusalem', '<', 'Israel', '<', 'Jew', '<', 'Judaism', 'Hebrew_calendar']
['Citizenship', 'Nationalism', 'Hebrew_language', 'Israel', '<', 'Jew', '<', 'Judaism', 'Hebrew_calendar']
['Citizenship', 'Nationalism', 'Hebrew_language', 'Jew', '<', 'Judaism', 'Hebrew_ca

(['Citizenship',
  'Nationalism',
  'Hebrew_language',
  'Judaism',
  'Hebrew_calendar'],
 1,
 'Sucess')

In [67]:
model_outputs_df = pd.read_csv('../logs/inference/runs/2023-09-28_17-13-19-996276/Llama-2-7b-chat-hf_outputs.csv')
print(model_outputs_df['model_output'][0])

reme_Court", 
  "Target": "Habeas_corpus"
} [/INST{
  "Path": "The_Supreme_Court;United_States;Law;Judiciary;Habeas_corpus"
}
